# OTS-SC-DCOPF — Solution quality (improving the K sweep)

**Where this fits.** Analysis A runs `../ots-sc-dcopf.ipynb` for K = 0, …, 10 (via
`../sweep_k.py`). For K ≥ 3 Gurobi stops at the time limit, so the saved solutions are the *best
found*, not proven optimal. This notebook takes those saved solutions and asks two questions:

1. **Is there a better topology nearby?** A local search that tests small changes to the saved
   topology (open one more line, or swap an open line for a closed one).
2. **Which contingencies limit the cost?** For the final topologies, which lines reach their
   rating in which states (the "limiting contingencies" metric of the thesis).

Improved solutions are saved in `../results/.../contingencies_improved/K<k>/`, in the same format
as the sweep. `../ots-sc-dcopf.ipynb` also looks in that folder for its warm start, so re-running
a K with `sweep_k.py` starts from the improved solution.

**The model is the same as in `../ots-sc-dcopf.ipynb`** (same equations, data and exclusions).
If the model changes there, change it here too. Section 6 checks that both give the same costs.

### The key idea: with the topology fixed, the OTS is an LP

The OTS is hard because of the binary variables $z_{ij}$. Once every $z_{ij}$ is fixed to 0 or
1, nothing binary is left: the problem is the SC-DCOPF on that topology, a linear program that
Gurobi solves in about 1.5 s for all 194 states. So we can **evaluate hundreds of topologies**
quickly, and pick the cheapest. This does not prove optimality (we only look at topologies close
to the one we have), but it can find better solutions than the MILP found in its time limit.

## 0. Settings

In [ ]:
FORMULATION = "redispatch_with_exclusions"   # Analysis A
CONFIG      = "contingencies"                # base + C_T + C_G

SEARCH_K          = [3, 4, 5, 6, 7, 8, 9, 10]   # K values to improve (K = 0, 1, 2 are proven optimal)
SEARCH_BUDGET_S   = 1800                        # [s] time limit of the local search per K
TOP_CANDIDATES    = 30                          # swaps only try the 30 most promising lines to open
REPORT_K          = [0, 2, 3, 4, 8]             # K values for the limiting-contingency report
THREADS           = 0                           # Gurobi threads (0 = all cores)

## 1. Data

Same processed data as `../ots-sc-dcopf.ipynb`. This notebook lives one folder deeper
(`solution-quality/`), so the paths start with `../..` (data) and `../` (results).

In [ ]:
import os
import time

import numpy as np
import pandas as pd
import gurobipy as gp
from gurobipy import GRB

ROOT   = "../.."
SOURCE = "matpower"
PROC   = f"{ROOT}/processed/{SOURCE}"

gens     = pd.read_csv(f"{PROC}/gens.csv")
loads    = pd.read_csv(f"{PROC}/loads.csv")
branches = pd.read_csv(f"{PROC}/branches.csv")
meta     = pd.read_csv(f"{PROC}/system_meta.csv").set_index("key")["value"]

BASE_MVA  = float(meta["base_mva"])
SLACK_BUS = int(meta["slack_bus"])

SWEEP_DIR    = f"../results/{SOURCE}/{FORMULATION}/{CONFIG}"            # read: the K sweep
IMPROVED_DIR = f"../results/{SOURCE}/{FORMULATION}/{CONFIG}_improved"   # write: better solutions

ENV = gp.Env(params={"OutputFlag": 0})   # one Gurobi environment, logs off

print(f"base_mva = {BASE_MVA:.0f} MVA | slack bus = {SLACK_BUS}")
print(f"generators = {len(gens)} | buses = {len(loads)} | branches = {len(branches)}")

## 2. Sets and parameters

Same as `../ots-sc-dcopf.ipynb`: $F^{em} = 1.15 F^{\max}$, angle limits ±0.6 rad, per-branch big-M
$M_{ij} = (\theta^{\max} - \theta^{\min})/x_{ij}$, switchable set $\mathcal{L}^S$ = non-radial
branches.

In [ ]:
set_N  = loads["bus"].tolist()
set_G  = gens["gen_id"].tolist()
set_L  = branches["branch_id"].tolist()
set_LS = branches.loc[~branches["is_bridge"], "branch_id"].tolist()   # switchable (non-radial)

par_pd    = dict(zip(loads["bus"],          loads["pd_pu"]))
par_g_bus = dict(zip(gens["gen_id"],        gens["bus"]))
par_pmin  = dict(zip(gens["gen_id"],        gens["pmin_pu"]))
par_pmax  = dict(zip(gens["gen_id"],        gens["pmax_pu"]))
par_cost  = dict(zip(gens["gen_id"],        gens["c1"]))
par_from  = dict(zip(branches["branch_id"], branches["from_bus"]))
par_to    = dict(zip(branches["branch_id"], branches["to_bus"]))
par_x     = dict(zip(branches["branch_id"], branches["x_pu"]))
par_rate  = dict(zip(branches["branch_id"], branches["rate_pu"]))
par_label = dict(zip(branches["branch_id"], branches["label"]))

EMERG_FACTOR = 1.15
par_rate_em  = {l: EMERG_FACTOR * par_rate[l] for l in set_L}
THETA_MAX    = 0.6
THETA_MIN    = -THETA_MAX
par_M        = {l: (THETA_MAX - THETA_MIN) / par_x[l] for l in set_L}

set_gens_at   = {i: [] for i in set_N}
set_out_lines = {i: [] for i in set_N}
set_in_lines  = {i: [] for i in set_N}
for g in set_G:
    set_gens_at[par_g_bus[g]].append(g)
for l in set_L:
    set_out_lines[par_from[l]].append(l)
    set_in_lines[par_to[l]].append(l)

print(f"|N| = {len(set_N)} | |G| = {len(set_G)} | |L| = {len(set_L)} | |L^S| = {len(set_LS)}")

## 3. Operating states

Same states and ids as `../ots-sc-dcopf.ipynb` with `FORMULATION = "redispatch_with_exclusions"`:
base case, non-radial line outages, generator outages, minus line 8-5, line 38-65 and the
generator at bus 10. Same participation factors $\alpha_{g,c}$.

In [ ]:
EXCLUDE = [("line", 7), ("line", 95), ("gen", 0)]   # 8-5, 38-65, generator at bus 10

scenarios = [{"id": 0, "type": "base", "out_line": None, "out_gen": None, "desc": "base case"}]
for l in set_LS:
    if ("line", l) not in EXCLUDE:
        scenarios.append({"id": len(scenarios), "type": "line", "out_line": l, "out_gen": None,
                          "desc": f"line {par_label[l]} out"})
for g in set_G:
    if ("gen", g) not in EXCLUDE:
        scenarios.append({"id": len(scenarios), "type": "gen", "out_line": None, "out_gen": g,
                          "desc": f"gen {g} (bus {par_g_bus[g]}) out"})
scen  = {s["id"]: s for s in scenarios}
set_C = list(scen)

def line_on(l, c):
    return 0 if scen[c]["out_line"] == l else 1

def gen_on(g, c):
    return 0 if scen[c]["out_gen"] == g else 1

par_alpha = {}
for c in set_C:
    if scen[c]["type"] == "gen":
        cap_on = sum(gen_on(h, c) * par_pmax[h] for h in set_G)
        for g in set_G:
            if gen_on(g, c):
                par_alpha[(g, c)] = par_pmax[g] / cap_on

print(f"|C| = {len(set_C)} operating states")

## 4. The model

The same equations as `../ots-sc-dcopf.ipynb`, built once for all 194 states:

| Block | Equation |
|---|---|
| Objective | $\min \sum_g c_g P_{g,0} S_{base}$ |
| Energy balance (3.9) | $\sum_{g \in \mathcal{G}_i} P_{g,c} - P_{d,i} = \sum_{\Omega_i^+} f_{ij,c} - \sum_{\Omega_i^-} f_{ji,c}$ |
| Generator limits (3.10) | $P_g^{\min} N1_{g,c} \le P_{g,c} \le P_g^{\max} N1_{g,c}$ |
| Preventive coupling | $P_{g,c} = P_{g,0}$ for line contingencies |
| Generator response | $P_{g,c} = P_{g,0} + \alpha_{g,c} \sum_h (1 - N1_{h,c}) P_{h,0}$ |
| Thermal limits | $\lvert f_{ij,c} \rvert \le F_{ij} z_{ij}$ ($F^{\max}$ base case, $F^{em}$ contingencies) |
| DC flow (3.13)–(3.14) | $\lvert f_{ij,c} - (\theta_{i,c} - \theta_{j,c})/x_{ij} \rvert \le M_{ij}(1 - z_{ij})$ |
| Outaged branch (3.15) | $f_{ij,c} = 0$ if $N1_{ij,c} = 0$ |
| Angle limits, reference (3.16) | $\theta^{\min} \le \theta_{i,c} \le \theta^{\max}$, $\theta_{ref,c} = 0$ |

Two writing details that do not change the model: the angle limits are variable bounds instead
of constraints, and the thermal constraints get names (`fmax[l,c]`, `fmin[l,c]`) so that section 8
can find which ones are binding.

The binary $z_{ij}$ are in the model, but we never let Gurobi choose them: `evaluate()` fixes
each one (lower bound = upper bound = 0 or 1), so every solve is an LP.

In [ ]:
t0 = time.time()
model = gp.Model("ots_fixed_topology", env=ENV)
model.Params.Threads = THREADS
model.Params.DualReductions = 0   # report INFEASIBLE (not INF_OR_UNBD) for an infeasible topology

var_p     = model.addVars(set_G, set_C, lb=-GRB.INFINITY, name="p")
var_f     = model.addVars(set_L, set_C, lb=-GRB.INFINITY, name="f")
var_theta = model.addVars(set_N, set_C, lb=THETA_MIN, ub=THETA_MAX, name="theta")
var_z     = model.addVars(set_L, vtype=GRB.BINARY, name="z")

model.setObjective(gp.quicksum(par_cost[g] * var_p[g, 0] * BASE_MVA for g in set_G), GRB.MINIMIZE)

for c in set_C:
    kind = scen[c]["type"]
    for i in set_N:                                                   # energy balance
        model.addConstr(gp.quicksum(var_p[g, c] for g in set_gens_at[i]) - par_pd[i]
                        == gp.quicksum(var_f[l, c] for l in set_out_lines[i])
                        - gp.quicksum(var_f[l, c] for l in set_in_lines[i]))
    for g in set_G:                                                   # generator limits
        model.addConstr(var_p[g, c] >= par_pmin[g] * gen_on(g, c))
        model.addConstr(var_p[g, c] <= par_pmax[g] * gen_on(g, c))
    if kind == "line":                                                # preventive coupling
        for g in set_G:
            model.addConstr(var_p[g, c] == var_p[g, 0])
    if kind == "gen":                                                 # generator response
        lost = gp.quicksum((1 - gen_on(h, c)) * var_p[h, 0] for h in set_G)
        for g in set_G:
            if gen_on(g, c):
                model.addConstr(var_p[g, c] == var_p[g, 0] + par_alpha[(g, c)] * lost)
    rate = par_rate if c == 0 else par_rate_em
    for l in set_L:
        i, j = par_from[l], par_to[l]
        model.addConstr(var_f[l, c] >= -rate[l] * var_z[l], name=f"fmin[{l},{c}]")   # thermal
        model.addConstr(var_f[l, c] <=  rate[l] * var_z[l], name=f"fmax[{l},{c}]")
        if line_on(l, c):                                             # big-M DC flow
            dc_law = var_f[l, c] - (var_theta[i, c] - var_theta[j, c]) / par_x[l]
            model.addConstr(dc_law + par_M[l] * (1 - var_z[l]) >= 0)
            model.addConstr(dc_law - par_M[l] * (1 - var_z[l]) <= 0)
        else:                                                         # outaged branch
            model.addConstr(var_f[l, c] == 0)
    model.addConstr(var_theta[SLACK_BUS, c] == 0)                     # reference bus
model.update()
print(f"Model built in {time.time() - t0:.0f} s: {model.NumConstrs:,} constraints, "
      f"{model.NumVars:,} variables")

## 5. Evaluating a topology

A topology is written as the **set of open branches** (branch ids). `evaluate(open_set)` fixes
$z = 0$ on those branches and $z = 1$ on all the others, solves, and returns the cost in $/h, or
`None` if no dispatch satisfies all 194 states with that topology.

In [ ]:
def evaluate(open_set):
    # cost [$/h] of the SC-DCOPF with the branches in open_set opened, or None if infeasible
    for l in set_L:
        var_z[l].LB = var_z[l].UB = (0 if l in open_set else 1)
    model.optimize()
    return model.ObjVal if model.Status == GRB.OPTIMAL else None

def labels(open_set):
    return ", ".join(par_label[l] for l in sorted(open_set))   # in branch-id order

## 6. Check: does this reproduce the K sweep?

Before searching, we read every saved result of the sweep (`line_loading.csv` gives the topology
in column `z`, `summary.csv` gives the cost) and evaluate the same topology here.

- For K = 0, 1, 2 (proven optimal) the costs must be equal.
- For K ≥ 3 they should also be equal. If this LP is *lower*, the MILP stopped with a dispatch
  that was not the best one for its own topology (possible at a time limit). It must never be
  *higher*: that would mean the two notebooks do not have the same model.

In [ ]:
def load_result(folder):
    # (cost, open_set) saved in a result folder, or None
    if not os.path.exists(f"{folder}/summary.csv"):
        return None
    s  = pd.read_csv(f"{folder}/summary.csv").set_index("metric")["value"]
    ll = pd.read_csv(f"{folder}/line_loading.csv")
    return float(s["base_cost_usd_per_h"]), set(ll.loc[ll["z"] == 0, "branch_id"])

sweep = {}
for K in range(0, 11):
    r = load_result(f"{SWEEP_DIR}/K{K}")
    if r is not None:
        sweep[K] = {"cost": r[0], "open": r[1]}

rows = []
for K, s in sweep.items():
    t0 = time.time()
    s["cost_lp"] = evaluate(s["open"])
    rows.append({"K": K, "saved cost": s["cost"], "LP cost": s["cost_lp"],
                 "difference": s["cost_lp"] - s["cost"], "LP time (s)": round(time.time() - t0, 1),
                 "open": labels(s["open"])})
check = pd.DataFrame(rows)
display(check.round(2))

assert (check["difference"] <= 0.01).all(), "LP above the saved cost: the models differ"
assert (check.loc[check["K"] <= 2, "difference"].abs() <= 0.01).all(), "proven optima not reproduced"
print("OK: same costs as the K sweep.")

## 7. Local search

For each K in `SEARCH_K`, start from the **best topology known with at most K open lines** (the
sweep results for K' ≤ K, and what this search already found for smaller K), then repeat:

1. **Rank the closed lines.** For every closed switchable line $b$, evaluate the topology with
   $b$ opened *in addition* to the current ones (≈ 175 LPs). This tells how much opening $b$
   helps on its own. Lines that help a lot are the best candidates to bring into the topology.
2. **Add.** If fewer than K lines are open and the best line from step 1 lowers the cost, open it.
3. **Swap.** Otherwise (K lines already open), try closing one open line $a$ and opening one
   closed line $b$, taking $b$ from the ranking of step 1 (best first, only the
   `TOP_CANDIDATES` best). The first swap that lowers the cost is accepted.

After an add or a swap, the search goes back to step 1 from the new topology. It stops when
nothing lowers the cost (a *local optimum*: no better topology one step away, among the
candidates tried) or when `SEARCH_BUDGET_S` runs out.

Why the ranking: without it, the swaps are tried in branch-id order, and a good swap may be at the
end of a list of ~500. With it, the lines that are known to help are tried first. Example: the
better K = 3 solution comes from swapping 92-94 for 68-69; 68-69 is the line that helps most when
added to the K = 3 topology.

Every topology tried still satisfies all 194 states (`evaluate` checks them all), so every
improvement is a valid solution of the full problem.

Cost: each evaluation takes ~1–2 s. One pass is ≈ 175 evaluations for the ranking plus at most
`TOP_CANDIDATES` × K swaps (≈ 4–10 min).

In [ ]:
def local_search(K, start_open, start_cost, budget_s):
    cur_open, cur_cost = set(start_open), start_cost
    t_start, n_eval, history = time.time(), 0, []

    def out_of_time():
        return time.time() - t_start > budget_s

    def accept(desc, cand, cost):
        nonlocal cur_open, cur_cost
        print(f"  K={K}: {desc}: ${cur_cost:,.2f} -> ${cost:,.2f}/h "
              f"(after {n_eval} evaluations, {time.time() - t_start:.0f} s)", flush=True)
        history.append({"K": K, "move": desc, "from": cur_cost, "to": cost, "evaluations": n_eval})
        cur_open, cur_cost = cand, cost

    while True:
        # 1. rank the closed lines: cost with that line opened in addition to the current ones
        closed, add_cost = [l for l in set_LS if l not in cur_open], {}
        for b in closed:
            if out_of_time():
                return cur_open, cur_cost, history, n_eval, "time budget"
            add_cost[b] = evaluate(cur_open | {b})
            n_eval += 1
        ranked = sorted((b for b in closed if add_cost[b] is not None), key=lambda b: add_cost[b])

        # 2. add: open the best line, if there is room and it helps
        if len(cur_open) < K and ranked and add_cost[ranked[0]] < cur_cost - 0.01:
            b = ranked[0]
            accept(f"open {par_label[b]}", cur_open | {b}, add_cost[b])
            continue

        # 3. swap: close a, open b (b from the ranking, best first)
        improved = False
        for b in ranked[:TOP_CANDIDATES]:
            for a in sorted(cur_open):
                if out_of_time():
                    return cur_open, cur_cost, history, n_eval, "time budget"
                cand = (cur_open - {a}) | {b}
                cost = evaluate(cand)
                n_eval += 1
                if cost is not None and cost < cur_cost - 0.01:
                    accept(f"close {par_label[a]}, open {par_label[b]}", cand, cost)
                    improved = True
                    break
            if improved:
                break
        if not improved:
            return cur_open, cur_cost, history, n_eval, "local optimum"

Run the search and keep the best topology of each K. Results are shown at the end of each K.

In [ ]:
best = {K: (s["cost_lp"], s["open"]) for K, s in sweep.items()}   # best known per K
search_rows, moves = [], []

for K in SEARCH_K:
    # start: cheapest known topology with at most K open lines
    start_cost, start_open = min((best[k] for k in best if k <= K), key=lambda t: t[0])
    print(f"\nK = {K}: start ${start_cost:,.2f}/h, open [{labels(start_open)}]")
    t0 = time.time()
    new_open, new_cost, hist, n_eval, why = local_search(K, start_open, start_cost, SEARCH_BUDGET_S)
    best[K] = (new_cost, new_open)
    moves += hist
    search_rows.append({"K": K, "sweep cost": sweep[K]["cost"] if K in sweep else None,
                        "start cost": start_cost, "best cost": new_cost,
                        "gain vs sweep": (new_cost - sweep[K]["cost"]) if K in sweep else None,
                        "stopped by": why, "evaluations": n_eval,
                        "time (s)": round(time.time() - t0), "open": labels(new_open)})
    print(f"K = {K}: {why}, ${new_cost:,.2f}/h after {n_eval} evaluations")

search = pd.DataFrame(search_rows)
display(search.round(2))

### 7.1 Save the improved solutions

For every K whose cost went down compared with the sweep, save the solution in
`../results/.../contingencies_improved/K<k>/`:

- `summary.csv` — same metric names as the sweep (`base_cost_usd_per_h`, …) plus
  `source = local search`;
- `line_loading.csv` — the topology (`z`) and the base-case flows, same columns as the sweep.

`../ots-sc-dcopf.ipynb` looks in this folder when choosing its warm start. To let Gurobi try to
improve a K further from here: `python sweep_k.py --k-min K --k-max K --time-limit 9000` from
`ots-sc-dcopf/`.

In [ ]:
saved_k = []
for K in SEARCH_K:
    cost, open_set = best[K]
    if K in sweep and cost >= sweep[K]["cost"] - 0.01:
        continue                                    # not better than the sweep: nothing to save
    assert evaluate(open_set) is not None           # re-solve to read the flows of this topology
    out = f"{IMPROVED_DIR}/K{K}"
    os.makedirs(out, exist_ok=True)
    pd.DataFrame({"metric": ["base_cost_usd_per_h", "max_switched", "n_switched_lines", "source"],
                  "value": [round(model.ObjVal, 2), K, len(open_set), "local search"]}
                 ).to_csv(f"{out}/summary.csv", index=False)
    rows = []
    for l in set_L:
        f_mw = var_f[l, 0].X * BASE_MVA
        rows.append({"branch_id": l, "label": par_label[l], "z": 0 if l in open_set else 1,
                     "f_mw": round(f_mw, 2), "fmax_mw": round(par_rate[l] * BASE_MVA, 1),
                     "loading_pct": round(100 * abs(f_mw) / (par_rate[l] * BASE_MVA), 1)})
    pd.DataFrame(rows).sort_values("loading_pct", ascending=False).to_csv(
        f"{out}/line_loading.csv", index=False)
    saved_k.append(K)
print(f"Saved improved solutions for K = {saved_k} in {IMPROVED_DIR}/" if saved_k
      else "No K improved: nothing saved.")

## 8. Limiting contingencies

Which contingencies actually limit the cost? We use the **dual values** (shadow prices) of the
thermal constraints.

- After solving an LP, every constraint has a dual value. For a thermal constraint
  $f_{ij,c} \le F_{ij}$, the dual says **how much the cost would drop if that line had one more
  MW of capacity in that state**. Gurobi gives it in $/h per p.u. (our flows are in p.u.);
  dividing by $S_{base}$ = 100 MVA gives $/h per MW, i.e. **$/MWh**.
- A dual of zero means the constraint is not binding: the line is below its limit, or at it by
  chance without affecting the cost.
- A non-zero dual means **this line, in this state, is what stops the dispatch from being
  cheaper**. Those states are the limiting contingencies.

Gurobi only gives duals for LPs. Our model has (fixed) binary variables, so it is formally a MIP.
`model.fixed()` returns the same model with the binaries replaced by their fixed values, which is
a pure LP with duals.

For each K in `REPORT_K` (best topology), the table lists every binding (state, line) pair. The
second table sums the duals per state, to rank the contingencies.

In [ ]:
def binding_constraints(open_set):
    # [(state, branch, |dual| in $/MWh)] of the binding thermal constraints
    assert evaluate(open_set) is not None
    fixed = model.fixed()                            # same LP, binaries removed -> duals available
    fixed.optimize()
    out = []
    for con in fixed.getConstrs():
        name = con.ConstrName
        if name.startswith(("fmax[", "fmin[")) and abs(con.Pi) / BASE_MVA > 0.01:   # > 0.01 $/MWh
            l, c = map(int, name[5:-1].split(","))
            out.append((c, l, abs(con.Pi) / BASE_MVA))   # $/h per p.u. -> $/MWh
    fixed.dispose()
    return out

rows = []
for K in REPORT_K:
    cost, open_set = best[K]
    for c, l, dual in binding_constraints(open_set):
        rows.append({"K": K, "state": c, "contingency": scen[c]["desc"], "line at limit": par_label[l],
                     "dual ($/MWh)": round(dual, 1)})
limiting = pd.DataFrame(rows)
display(limiting.sort_values(["K", "dual ($/MWh)"], ascending=[True, False]))

In [ ]:
# Ranking of the limiting contingencies: sum of duals per state, one column per K
ranking = (limiting.pivot_table(index="contingency", columns="K", values="dual ($/MWh)",
                                aggfunc="sum", fill_value=0)
           .sort_values(REPORT_K[0], ascending=False))
display(ranking.round(1))
print(f"Limiting states per K: "
      + ", ".join(f"K={K}: {limiting.loc[limiting['K'] == K, 'state'].nunique()}" for K in REPORT_K))

## 9. Summary

Final table: the cost per K from the sweep and after the local search, and the saving with
respect to K = 0 (no switching).

In [ ]:
c0 = best[0][0]
summary = pd.DataFrame([{"K": K,
                         "sweep cost": sweep[K]["cost"] if K in sweep else None,
                         "best cost": best[K][0],
                         "saving vs K0 (%)": 100 * (c0 - best[K][0]) / c0,
                         "n open": len(best[K][1]),
                         "open": labels(best[K][1])} for K in sorted(best)])
display(summary.round(2))